# RoadSafe Delhi — Support Vector Machine (SVM) Model\n\n**Model development, kernel comparison, validation threshold selection, final evaluation, and artifact saving.**

## 1. Imports and project paths\nLoad libraries, locate the repository, and load the official spatial partitions.

In [ ]:
# Import the libraries used for data loading, preprocessing, SVM training,
# evaluation, visualization, saving model files, and reproducibility checks.
import os
import sys
import json
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, precision_recall_curve,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)

print("All libraries imported successfully!")

## 2. Target and approved feature selection\nLoad the feature contract and assert that forbidden predictors are excluded.

In [ ]:
# Locate the cloned RoadSafe-Delhi repository and its official data_final folder.
# This supports running the notebook from the repository root or models/svm/notebooks.

current_dir = Path.cwd()
possible_roots = [current_dir, current_dir / "RoadSafe-Delhi", *current_dir.parents]
PROJECT_ROOT = next(
    (path.resolve() for path in possible_roots
     if (path / "data_final" / "feature_contract.json").exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate RoadSafe-Delhi/data_final/feature_contract.json. "
        "Open this notebook from the cloned RoadSafe-Delhi project."
    )

DATA_FINAL_DIR = PROJECT_ROOT / "data_final"
ARTIFACTS_DIR = PROJECT_ROOT / "models" / "svm" / "artifacts"
OUTPUTS_DIR = PROJECT_ROOT / "models" / "svm" / "outputs"
SRC_DIR = PROJECT_ROOT / "models" / "svm" / "src"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

# Load the common training, validation, and test spatial partitions.
train_df = pd.read_parquet(DATA_FINAL_DIR / "train.parquet")
val_df = pd.read_parquet(DATA_FINAL_DIR / "validation.parquet")
test_df = pd.read_parquet(DATA_FINAL_DIR / "test.parquet")

print("Project root:", PROJECT_ROOT)
print("Training dataset:", train_df.shape)
print("Validation dataset:", val_df.shape)
print("Test dataset:", test_df.shape)

## 3. Spatial split verification\nCheck that train, validation, and test spatial blocks are pairwise disjoint.

In [ ]:
# Set the required target and load approved predictors from the project's feature contract.
# Check that no quarantined or target-related columns are included as model inputs.

TARGET = "is_crash_segment"

with open(DATA_FINAL_DIR / "feature_contract.json", "r", encoding="utf-8") as file:
    feature_contract = json.load(file)

NUMERIC = feature_contract["approved_numerical_predictors"]
CATEGORICAL = feature_contract["approved_categorical_predictors"]
FEATURES = NUMERIC + CATEGORICAL

forbidden_features = (
    set(feature_contract["quarantined_leakage_columns"])
    | set(feature_contract["target_definitions"].keys())
)

assert TARGET in train_df.columns, f"Target column {TARGET} is missing."
assert TARGET not in FEATURES, "The target cannot be an input feature."
assert not (set(FEATURES) & forbidden_features), "Forbidden or leakage feature detected."

for name, dataset in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    missing = set(FEATURES + [TARGET]) - set(dataset.columns)
    assert not missing, f"{name} dataset is missing columns: {missing}"

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val, y_val = val_df[FEATURES], val_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

print("Target:", TARGET)
print("Approved features:", FEATURES)
print("Feature-contract leakage checks passed.")

## 4. Preprocessing\nConfigure numerical scaling and categorical one-hot encoding, fitted on training data only.

In [ ]:
# Verify that no spatial block is shared by any pair of the three official data partitions.
# This checks spatial split separation, but does not claim to prove every possible form of leakage is absent.

train_blocks = set(train_df["spatial_block"])
val_blocks = set(val_df["spatial_block"])
test_blocks = set(test_df["spatial_block"])

train_val_overlap = train_blocks & val_blocks
train_test_overlap = train_blocks & test_blocks
val_test_overlap = val_blocks & test_blocks

print("TRAIN - VALIDATION overlap:", len(train_val_overlap))
print("TRAIN - TEST overlap:", len(train_test_overlap))
print("VALIDATION - TEST overlap:", len(val_test_overlap))

assert not train_val_overlap
assert not train_test_overlap
assert not val_test_overlap

spatial_verification = {
    "train_validation_overlap": len(train_val_overlap),
    "train_test_overlap": len(train_test_overlap),
    "validation_test_overlap": len(val_test_overlap),
    "all_pairwise_spatial_blocks_disjoint": True
}
print("All three datasets have separate spatial blocks.")

## 5. SVM kernel comparison\nTrain linear, RBF, and polynomial SVM pipelines.

In [ ]:
# Configure numerical scaling and categorical one-hot encoding.
# The preprocessing steps are fitted as part of the model pipeline using training data only.

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)
])

print("Preprocessing pipeline configured.")

## 6. Validation kernel comparison\nCompare the trained kernels using validation metrics.

In [ ]:
# Train the existing linear, RBF, and polynomial SVM kernel models.
# Probability estimates are enabled so the selected model can use the required validation threshold procedure.

kernels = ["linear", "rbf", "poly"]
models = {}

for kernel in kernels:
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", SVC(
            kernel=kernel,
            C=1,
            gamma="scale",
            class_weight="balanced",
            probability=True,
            random_state=42,
            cache_size=1000
        ))
    ])
    model.fit(X_train, y_train)
    models[kernel] = model
    print(kernel, "SVM trained successfully!")

## 7. Hyperparameter search\nRetain the existing kernel and C search using validation PR-AUC, then F1.

In [ ]:
# Compare the trained kernels on the validation partition.
# These validation results support model selection; the test partition is not used here.

results = []
for kernel, model in models.items():
    predictions = model.predict(X_val)
    probabilities = model.predict_proba(X_val)[:, 1]
    results.append({
        "Kernel": kernel,
        "Accuracy": accuracy_score(y_val, predictions),
        "Precision": precision_score(y_val, predictions, zero_division=0),
        "Recall": recall_score(y_val, predictions, zero_division=0),
        "F1 Score": f1_score(y_val, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_val, probabilities),
        "PR-AUC": average_precision_score(y_val, probabilities)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

## 8. Training performance check\nReview training performance for comparison with validation and test results.

In [ ]:
# Search the same kernel and C combinations used in the previous notebook.
# Select by validation PR-AUC, with validation F1 as the tie-breaker.
# The held-out test data is not used during this search.

best_model = None
best_score = -1
best_params = {}

for kernel in ["linear", "rbf", "poly"]:
    for C in [0.1, 1, 10, 100]:
        params = {
            "kernel": kernel,
            "C": C,
            "class_weight": "balanced",
            "probability": True,
            "random_state": 42
        }
        if kernel in ["rbf", "poly"]:
            params["gamma"] = "scale"

        candidate_model = Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", SVC(**params, cache_size=1000))
        ])
        candidate_model.fit(X_train, y_train)

        val_probabilities = candidate_model.predict_proba(X_val)[:, 1]
        val_predictions = candidate_model.predict(X_val)
        pr_auc = average_precision_score(y_val, val_probabilities)
        f1 = f1_score(y_val, val_predictions, zero_division=0)

        if (pr_auc, f1) > (best_score, best_params.get("f1", -1)):
            best_score = pr_auc
            best_model = candidate_model
            best_params = {
                "kernel": kernel,
                "C": C,
                "gamma": params.get("gamma", "scale"),
                "class_weight": "balanced",
                "probability": True,
                "random_state": 42,
                "validation_f1_at_default_threshold": float(f1),
                "validation_pr_auc": float(pr_auc)
            }

print("Best validation configuration:", best_params)

## 9. Validation threshold selection\nChoose the threshold that maximizes validation F1 and freeze it before testing.

In [ ]:
# Measure the selected model on the training partition.
# Compare these values with validation and test performance when reviewing possible overfitting.

train_pred = best_model.predict(X_train)
train_probabilities = best_model.predict_proba(X_train)[:, 1]

print("TRAINING PERFORMANCE")
print("Accuracy:", accuracy_score(y_train, train_pred))
print("Precision:", precision_score(y_train, train_pred, zero_division=0))
print("Recall:", recall_score(y_train, train_pred, zero_division=0))
print("F1 Score:", f1_score(y_train, train_pred, zero_division=0))
print("PR-AUC:", average_precision_score(y_train, train_probabilities))

## 10. Final test evaluation\nEvaluate once on the untouched test partition using the frozen threshold.

In [ ]:
# Select the classification threshold using validation probabilities only.
# Choose the threshold with the highest validation F1, then freeze it before test evaluation.

y_val_prob = best_model.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_prob)

validation_f1_scores = np.array([
    f1_score(y_val, (y_val_prob >= threshold).astype(int), zero_division=0)
    for threshold in thresholds
])

best_threshold_index = int(np.argmax(validation_f1_scores))
optimal_threshold = float(thresholds[best_threshold_index])
validation_threshold_f1 = float(validation_f1_scores[best_threshold_index])

print(f"Optimal validation threshold: {optimal_threshold:.6f}")
print(f"Validation F1 at selected threshold: {validation_threshold_f1:.4f}")

## 11. Test visualizations\nSave the confusion matrix, ROC curve, and precision-recall curve.

In [ ]:
# Evaluate the selected model once on the untouched test partition.
# Apply the threshold selected on validation data; do not tune it using test labels.

y_test_prob = best_model.predict_proba(X_test)[:, 1]
y_scores = best_model.decision_function(X_test)  # Retained from the previous workflow.
y_pred = (y_test_prob >= optimal_threshold).astype(int)

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1 Score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_test_prob),
    "PR-AUC": average_precision_score(y_test, y_test_prob),
    "Classification Threshold": optimal_threshold
}

print("FINAL SVM TEST RESULTS")
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

print("\\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

## 12. Support vector analysis\nReport the support vector count overall and per class.

In [ ]:
# Create and save the final test confusion matrix, ROC curve, and precision-recall curve.
# The plots use the final test predictions and probability scores.

os.makedirs(OUTPUTS_DIR, exist_ok=True)

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title(f"SVM Test Confusion Matrix (threshold={optimal_threshold:.3f})")
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "confusion_matrix.png", dpi=300)
plt.show()
plt.close()

RocCurveDisplay.from_predictions(y_test, y_test_prob)
plt.title("SVM Test ROC Curve")
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "roc_curve.png", dpi=300)
plt.show()
plt.close()

PrecisionRecallDisplay.from_predictions(y_test, y_test_prob)
plt.title("SVM Test Precision-Recall Curve")
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "precision_recall_curve.png", dpi=300)
plt.show()
plt.close()

## 13. Save model and preprocessing artifacts\nSave the full pipeline, separate scaler and encoder, threshold, metrics, and test files.

In [ ]:
# Report the number of support vectors used by the selected SVM classifier.
# For an RBF model, support vectors help define the nonlinear decision boundary.

svm_classifier = best_model.named_steps["classifier"]
support_vectors_per_class = svm_classifier.n_support_
total_support_vectors = int(support_vectors_per_class.sum())

print("Support vectors per class:", support_vectors_per_class)
print("Total support vectors:", total_support_vectors)
print("Classes:", svm_classifier.classes_)

## 14. Independent prediction reproduction\nRun a separate script using the saved model and threshold and compare predictions.

In [ ]:
# Save the complete trained pipeline and the separately requested fitted preprocessing objects.
# Also save the validation-selected threshold, configuration, metrics, spatial check, and test predictions.

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

fitted_preprocessor = best_model.named_steps["preprocessor"]
fitted_scaler = fitted_preprocessor.named_transformers_["num"]
fitted_encoder = fitted_preprocessor.named_transformers_["cat"]

joblib.dump(best_model, ARTIFACTS_DIR / "svm_model.joblib")
joblib.dump(fitted_scaler, ARTIFACTS_DIR / "scaler.joblib")
joblib.dump(fitted_encoder, ARTIFACTS_DIR / "encoder.joblib")

with open(ARTIFACTS_DIR / "threshold.json", "w", encoding="utf-8") as file:
    json.dump({"optimal_threshold": optimal_threshold}, file, indent=4)

with open(ARTIFACTS_DIR / "model_config.json", "w", encoding="utf-8") as file:
    json.dump(best_params, file, indent=4)

with open(OUTPUTS_DIR / "metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=4)

with open(OUTPUTS_DIR / "spatial_verification.json", "w", encoding="utf-8") as file:
    json.dump(spatial_verification, file, indent=4)

pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": y_pred,
    "Probability": y_test_prob,
    "Decision_Score": y_scores
}).to_csv(OUTPUTS_DIR / "test_predictions.csv", index=False)

X_test.to_csv(OUTPUTS_DIR / "test_features.csv", index=False)

print("Saved model, scaler, encoder, threshold, metrics, and test files.")
print("Artifacts:", ARTIFACTS_DIR)
print("Outputs:", OUTPUTS_DIR)

## 15. Reusable SVM module\nCreate reusable model loading, threshold, prediction, and evaluation functions.

In [ ]:
# Create a standalone script that reloads the saved model and threshold.
# It independently predicts the test set using the same frozen validation threshold.

reproduction_code = """
from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(__file__).resolve().parents[3]
ARTIFACTS_DIR = PROJECT_ROOT / "models" / "svm" / "artifacts"
OUTPUTS_DIR = PROJECT_ROOT / "models" / "svm" / "outputs"

model = joblib.load(ARTIFACTS_DIR / "svm_model.joblib")
with open(ARTIFACTS_DIR / "threshold.json", "r", encoding="utf-8") as file:
    threshold = json.load(file)["optimal_threshold"]

X_test = pd.read_csv(OUTPUTS_DIR / "test_features.csv")
probabilities = model.predict_proba(X_test)[:, 1]
predictions = (probabilities >= threshold).astype(int)

np.savetxt(OUTPUTS_DIR / "reproduced_predictions.csv",
           predictions, fmt="%d", delimiter=",")

print("Independent prediction completed!")
print("Total predictions:", len(predictions))
"""

with open(SRC_DIR / "reproduce.py", "w", encoding="utf-8") as file:
    file.write(reproduction_code)

print("Reproduction script created:", SRC_DIR / "reproduce.py")

## 16. Documentation and final checks\nWrite the README and evaluation report and verify the required files.

In [ ]:
# Run the independent reproduction script from the repository root.
# Display any error and stop if the script does not complete successfully.

reproduction_script = SRC_DIR / "reproduce.py"
result = subprocess.run(
    [sys.executable, str(reproduction_script)],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Independent reproduction failed.")

## 17. Package the SVM folder\nCreate a ZIP copy of the completed `models/svm` folder after all model and report files have been generated.

In [ ]:
# Package the completed SVM folder, including its notebook, artifacts, source files, and outputs.
# Run this cell only after the notebook has been saved inside models/svm/notebooks and all earlier cells succeed.

import shutil

zip_base = PROJECT_ROOT / "RoadSafe_SVM_Final"
zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=PROJECT_ROOT,
    base_dir="models/svm"
)

print("ZIP created successfully!")
print("ZIP location:", zip_path)